<a href="https://colab.research.google.com/github/Phani-ISB/Reconciliation_Modules/blob/main/BankRecon_H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Bank Reconciliation — Simple Working Version
### Ledger (`Ledger_InputData.xlsx`) ↔ Statement (`Statement_InputData.xlsx`)

Run cells **top to bottom**. Cell 3 & 4 show raw data so you can verify columns before any matching.


In [1]:
# ── CELL 1 | Installs ────────────────────────────────────────────────────────
# !pip install pandas openpyxl -q    # uncomment in Colab
import pandas as pd
import numpy as np
import re
print("✅ Ready  |  pandas", pd.__version__)


✅ Ready  |  pandas 2.2.2


In [2]:
# ── CELL 2 | File & Column Config ────────────────────────────────────────────
# ⚠ Only edit this block if your file paths or column names differ

LEDGER_FILE    = "/content/Ledger_InputData.xlsx"
LEDGER_SHEET   = 0                        # 0 = first sheet, or use sheet name

STATEMENT_FILE = "/content/Statement_InputData.xlsx"
STATEMENT_SHEET = 0

# Ledger columns
L_DATE        = "Posting Dt"
L_DESC        = "Description"
L_PARTICULARS = "Particulars"
L_DR_CR       = "Debit-Credit"           # e.g. "50000 Dr" or "30000 Cr"
L_CHEQUE      = "Cheque No"
L_DOC_NO      = "Doc No"
L_DOC_TYPE    = "Doc Type"
L_BANK_GL     = "Bank GL A/C"

# Statement columns
S_DATE        = "Date"
S_NARRATION   = "Narration"
S_NARRATION2  = "Narration 2"
S_CREDIT      = "Credit"                 # amount when money comes IN
S_DEBIT       = "Debit"                  # amount when money goes OUT
S_CR_DR_FLAG  = "Credit-Debit"           # "C" or "D" flag
S_BANK_REF    = "Bank Ref"
S_CUST_REF    = "Cust Ref"
S_UNIQUE_ID   = "Unique ID"

# Matching tolerance
DATE_RANGE_DAYS  = 3      # for "inside range" rules
PARTIAL_MIN_RATIO = 0.4   # fraction of words that must overlap (0–1)
AMOUNT_TOLERANCE  = 1.0   # ₹ tolerance on amount comparison

print("✅ Config set")


✅ Config set


In [3]:
# ── CELL 3 | Load Ledger — Inspect Raw Data ───────────────────────────────────
raw_led = pd.read_excel(LEDGER_FILE, sheet_name=LEDGER_SHEET, dtype=str)
raw_led.columns = raw_led.columns.str.strip()
raw_led = raw_led.apply(lambda c: c.str.strip() if c.dtype == "object" else c)

print("=== LEDGER ===")
print(f"Shape   : {raw_led.shape}")
print(f"Columns : {list(raw_led.columns)}")
print()
print("--- First 5 rows ---")
display(raw_led.head())
print()
print(f"--- '{L_DATE}' sample ---")
print(raw_led[L_DATE].head(10).tolist())
print()
print(f"--- '{L_DR_CR}' sample ---")
print(raw_led[L_DR_CR].head(10).tolist())


=== LEDGER ===
Shape   : (2977, 10)
Columns : ['Doc Type', 'Posting Dt', 'Doc No', 'Cheque No', 'Description', 'Particulars', 'Debit-Credit', 'Bank', 'GL A/C', 'GL A/c Description']

--- First 5 rows ---


,Doc Type,Posting Dt,Doc No,Cheque No,Description,Particulars,Debit-Credit,Bank,GL A/C,GL A/c Description
0,KZ,2021-03-26 00:00:00,601244,NaN,NaN,NaN,-2620.28,WELLS FARGO BANK-121000248,213005.0,SALARY PAYABLE - STAFF
1,KZ,2021-03-26 00:00:00,601244,NaN,NaN,NaN,-786.79,WELLS FARGO BANK-121000248,213005.0,SALARY PAYABLE - STAFF
2,KZ,2021-03-26 00:00:00,601244,NaN,NaN,NaN,-810.56,WELLS FARGO BANK-121000248,213005.0,SALARY PAYABLE - STAFF
3,KZ,2021-03-26 00:00:00,601244,NaN,NaN,NaN,-47.77,WELLS FARGO BANK-121000248,213005.0,SALARY PAYABLE - STAFF
4,KZ,2021-03-26 00:00:00,601244,NaN,NaN,NaN,-1039.42,WELLS FARGO BANK-121000248,213005.0,SALARY PAYABLE - STAFF



--- 'Posting Dt' sample ---
['2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-26 00:00:00', '2021-03-16 00:00:00', '2021-03-16 00:00:00']

--- 'Debit-Credit' sample ---
['-2620.28', '-786.79', '-810.56', '-47.77', '-1039.42', '-596.13', '-668.88', '-643.85', '-75', '-75']


In [4]:
# ── CELL 4 | Load Statement — Inspect Raw Data ───────────────────────────────
raw_stm = pd.read_excel(STATEMENT_FILE, sheet_name=STATEMENT_SHEET, dtype=str)
raw_stm.columns = raw_stm.columns.str.strip()
raw_stm = raw_stm.apply(lambda c: c.str.strip() if c.dtype == "object" else c)

print("=== STATEMENT ===")
print(f"Shape   : {raw_stm.shape}")
print(f"Columns : {list(raw_stm.columns)}")
print()
print("--- First 5 rows ---")
display(raw_stm.head())
print()
print(f"--- '{S_DATE}' sample ---")
print(raw_stm[S_DATE].head(10).tolist())
print()
print(f"--- '{S_CREDIT}' / '{S_DEBIT}' sample ---")
print(raw_stm[[S_CREDIT, S_DEBIT]].head(10).to_string())


=== STATEMENT ===
Shape   : (250, 11)
Columns : ['Date', 'Unique ID', 'Bank Ref', 'Cust Ref', 'Narration', 'Narration 2', 'Opening Flag', 'Credit-Debit', 'BANK', 'Credit', 'Debit']

--- First 5 rows ---


,Date,Unique ID,Bank Ref,Cust Ref,Narration,Narration 2,Opening Flag,Credit-Debit,BANK,Credit,Debit
0,2021-03-16 00:00:00,00000091005864264235,IA000013544358,00000000000,469 / MISCELLANEOUS ACH DEBIT,CT DOR PAYMENT BUS DIRPAY 210316 XXXXX05470007...,NaN,-1600,WELLS FARGO BANK-121000248,NaN,1600.0
1,2021-01-19 00:00:00,NaN,IA000012026893,00000000000,469 / MISCELLANEOUS ACH DEBIT,ACH ORIGINATION - NTL - FILE 7878782339 COID 1...,NaN,-539182.25,WELLS FARGO BANK-121000248,NaN,539182.25
2,2021-01-19 00:00:00,NaN,IA000016600398,00000000000,469 / MISCELLANEOUS ACH DEBIT,ACH ORIGINATION - NTL - FILE 7878782339 COID 1...,NaN,-12582.98,WELLS FARGO BANK-121000248,NaN,12582.98
3,2021-01-19 00:00:00,91004139350708,IA000096035078,00000000000,469 / MISCELLANEOUS ACH DEBIT,GA0676 COFORGE L 4THQTRTAX 210115 GA0676 COFOR...,NaN,-9550.73,WELLS FARGO BANK-121000248,NaN,9550.73
4,2021-01-19 00:00:00,91004139350706,IA000011603915,00000000000,469 / MISCELLANEOUS ACH DEBIT,NAVIA BENEFIT SO FLEXIBLE B 210118 NIT NIIT TE...,NaN,-5120,WELLS FARGO BANK-121000248,NaN,5120.0



--- 'Date' sample ---
['2021-03-16 00:00:00', '2021-01-19 00:00:00', '2021-01-19 00:00:00', '2021-01-19 00:00:00', '2021-01-19 00:00:00', '2021-01-19 00:00:00', '2021-01-19 00:00:00', '2021-01-19 00:00:00', '2021-03-11 00:00:00', '2021-03-10 00:00:00']

--- 'Credit' / 'Debit' sample ---
      Credit       Debit
0        NaN      1600.0
1        NaN   539182.25
2        NaN    12582.98
3        NaN     9550.73
4        NaN      5120.0
5        NaN     2465.98
6        NaN      654.94
7        NaN      183.84
8        NaN  1977629.02
9  2999975.0         NaN


In [5]:
# ── CELL 5 | Parse & Clean Both Files ────────────────────────────────────────

def normalise(text):
    """Lowercase, alphanumeric + spaces only, collapse whitespace."""
    import re
    t = re.sub(r"[^a-z0-9\s]", " ", str(text).lower())
    return re.sub(r"\s+", " ", t).strip()

def to_float(val):
    """Best-effort convert any value to float. Returns NaN on failure."""
    try:
        return float(str(val).replace(",", "").strip())
    except:
        return np.nan

# ─── Ledger ───────────────────────────────────────────────────────────────────
# 'Debit-Credit' column format: "50,000 Dr" / "30,000 Cr"
# Extract the number; keep Dr/Cr as a side label.
import re
led = raw_led.copy()
led["_date"]      = pd.to_datetime(led[L_DATE], dayfirst=True, errors="coerce")
led["_amount"]    = led[L_DR_CR].apply(lambda v: to_float(re.sub(r"[^0-9.,]", "", str(v))))
led["_side"]      = led[L_DR_CR].apply(lambda v: "Cr" if re.search(r"cr", str(v), re.I) else "Dr")
led["_narration"] = (led[L_DESC].fillna("") + " " + led[L_PARTICULARS].fillna("")).str.strip()
led["_norm_narr"] = led["_narration"].apply(normalise)
led["_matched"]   = False
led["_rule"]      = ""
led["_stmt_idx"]  = pd.NA

# ─── Statement ────────────────────────────────────────────────────────────────
# 'Credit-Debit' column — use it directly as a number.
# Positive = Credit, Negative = Debit (or whatever the bank exports).
stm = raw_stm.copy()
stm["_date"]      = pd.to_datetime(stm[S_DATE], dayfirst=True, errors="coerce")
stm["_amount"]    = stm[S_CR_DR_FLAG].apply(to_float).abs()   # absolute value for matching
stm["_side"]      = stm[S_CR_DR_FLAG].apply(
                        lambda v: "Cr" if to_float(v) >= 0 else "Dr"
                    )
stm["_narration"] = (stm[S_NARRATION].fillna("") + " " + stm[S_NARRATION2].fillna("")).str.strip()
stm["_norm_narr"] = stm["_narration"].apply(normalise)
stm["_matched"]   = False
stm["_rule"]      = ""
stm["_led_idx"]   = pd.NA

# ─── Print raw samples so you can verify parsing is correct ──────────────────
print(f"Ledger   '{L_DR_CR}' raw  → {raw_led[L_DR_CR].head(5).tolist()}")
print(f"Ledger   '_amount' parsed → {led['_amount'].head(5).tolist()}")
print(f"Ledger   '_side'   parsed → {led['_side'].head(5).tolist()}")
print()
print(f"Statement '{S_CR_DR_FLAG}' raw  → {raw_stm[S_CR_DR_FLAG].head(5).tolist()}")
print(f"Statement '_amount' parsed → {stm['_amount'].head(5).tolist()}")
print(f"Statement '_side'   parsed → {stm['_side'].head(5).tolist()}")
print()

# ─── Overlap check ───────────────────────────────────────────────────────────
led_amts = set(led["_amount"].dropna().round(2))
stm_amts = set(stm["_amount"].dropna().round(2))
common   = led_amts & stm_amts
print(f"Amounts in BOTH files : {len(common)}  ← must be > 0 for any matches")
print(f"Sample                : {sorted(list(common))[:10]}")


Ledger   'Debit-Credit' raw  → ['-2620.28', '-786.79', '-810.56', '-47.77', '-1039.42']
Ledger   '_amount' parsed → [2620.28, 786.79, 810.56, 47.77, 1039.42]
Ledger   '_side'   parsed → ['Dr', 'Dr', 'Dr', 'Dr', 'Dr']

Statement 'Credit-Debit' raw  → ['-1600', '-539182.25', '-12582.98', '-9550.73', '-5120']
Statement '_amount' parsed → [1600.0, 539182.25, 12582.98, 9550.73, 5120.0]
Statement '_side'   parsed → ['Dr', 'Dr', 'Dr', 'Dr', 'Dr']

Amounts in BOTH files : 153  ← must be > 0 for any matches
Sample                : [3.63, 16.98, 20.5, 24.6, 27.06, 30.74, 50.0, 65.0, 66.0, 73.72]


/tmp/ipykernel_2724/1540159635.py:21: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  led["_date"]      = pd.to_datetime(led[L_DATE], dayfirst=True, errors="coerce")
/tmp/ipykernel_2724/1540159635.py:34: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  stm["_date"]      = pd.to_datetime(stm[S_DATE], dayfirst=True, errors="coerce")


In [6]:
# ── CELL 6 | Match Helpers ────────────────────────────────────────────────────

def record_match(li, si, rule_name):
    """Write a confirmed match into both DataFrames."""
    led.at[li, "_matched"]  = True
    led.at[li, "_rule"]     = rule_name
    led.at[li, "_stmt_idx"] = si
    stm.at[si, "_matched"]  = True
    stm.at[si, "_rule"]     = rule_name
    stm.at[si, "_led_idx"]  = li

def amounts_close(a, b):
    """True if two amounts are within AMOUNT_TOLERANCE of each other."""
    try:
        return abs(float(a) - float(b)) <= AMOUNT_TOLERANCE
    except:
        return False

def word_overlap_ratio(a, b):
    """
    Fraction of words from the shorter string that appear in the longer.
    e.g. 'neft tcs payment' vs 'tcs neft ltd' → 2/3 ≈ 0.67
    """
    wa = set(str(a).split())
    wb = set(str(b).split())
    if not wa or not wb:
        return 0.0
    shorter = min(len(wa), len(wb))
    return len(wa & wb) / shorter

def get_open_led():
    return led[~led["_matched"]]

def get_open_stm():
    return stm[~stm["_matched"]]

def run_rule(rule_fn, rule_name):
    """Run one rule, print how many it matched, return count."""
    n = rule_fn(rule_name)
    bar = "█" * n
    print(f"  {rule_name:<55} {n:>4} match(es)  {bar}")
    return n

def reset_all():
    """Clear all match flags — call before re-running the pipeline."""
    for col in ["_matched", "_rule"]:
        led[col] = (False if col == "_matched" else "")
        stm[col] = (False if col == "_matched" else "")
    led["_stmt_idx"] = pd.NA
    stm["_led_idx"]  = pd.NA
    print(f"✅ Reset — {len(led)} ledger / {len(stm)} statement rows all open")

print("✅ Match helpers ready")


# ── Rule classification ───────────────────────────────────────────────────────
# Drives Status, MatchType and Comments columns across all three reports.

RULE_STATUS = {
    "R6  Narration Exact & Date Exact         [strictest]" : "Automatic",
    "R1  Narration Exact                                  " : "Automatic",
    "R3  Date Exact                                        " : "Automatic",
    "R8  Narration Exact & Date Inside Range              " : "Knockoff",
    "R7  Narration Partial & Date Exact                   " : "Knockoff",
    "R9  Narration Partial & Date Inside Range            " : "Knockoff",
    "R4  Date Exact + Inside Range                        " : "Knockoff",
    "R10 Narration Partial & Date Any Range               " : "Knockoff",
    "R2  Narration Partial                                " : "Knockoff",
    "R5  Date Any Range                    [loosest]      " : "Knockoff",
}

RULE_MATCH_TYPE = {
    "R6  Narration Exact & Date Exact         [strictest]" : "Exact Match",
    "R1  Narration Exact                                  " : "Narration Exact Match",
    "R3  Date Exact                                        " : "Date Exact Match",
    "R8  Narration Exact & Date Inside Range              " : "Narration Exact, Date Range Match",
    "R7  Narration Partial & Date Exact                   " : "Narration Partial, Date Exact Match",
    "R9  Narration Partial & Date Inside Range            " : "Narration Partial, Date Range Match",
    "R4  Date Exact + Inside Range                        " : "Date Range Match",
    "R10 Narration Partial & Date Any Range               " : "Narration Partial Match",
    "R2  Narration Partial                                " : "Narration Partial Match",
    "R5  Date Any Range                    [loosest]      " : "Date Any Range Match",
}

RULE_COMMENTS = {
    "R6  Narration Exact & Date Exact         [strictest]" : "TransactionDate matches with df2_TransactionDate, Narration matches with df2_Narration",
    "R1  Narration Exact                                  " : "Narration matches with df2_Narration",
    "R3  Date Exact                                        " : "TransactionDate matches with df2_TransactionDate",
    "R8  Narration Exact & Date Inside Range              " : "Narration matches with df2_Narration, TransactionDate within range of df2_TransactionDate",
    "R7  Narration Partial & Date Exact                   " : "TransactionDate matches with df2_TransactionDate, Narration partially matches df2_Narration",
    "R9  Narration Partial & Date Inside Range            " : "Narration partially matches df2_Narration, TransactionDate within range of df2_TransactionDate",
    "R4  Date Exact + Inside Range                        " : "TransactionDate within range of df2_TransactionDate",
    "R10 Narration Partial & Date Any Range               " : "Narration partially matches df2_Narration",
    "R2  Narration Partial                                " : "Narration partially matches df2_Narration",
    "R5  Date Any Range                    [loosest]      " : "TransactionDate within extended range of df2_TransactionDate",
}

def get_status(rule):     return RULE_STATUS.get(rule, "Knockoff")
def get_match_type(rule): return RULE_MATCH_TYPE.get(rule, "Partial Match")
def get_comments(rule):   return RULE_COMMENTS.get(rule, "")

print("✅ Rule classification maps ready")


✅ Match helpers ready
✅ Rule classification maps ready


In [7]:
# ── CELL 7 | RULE 1 — Narration Exact  [O(m+n) hash-join] ───────────────────
# Join on (_norm_narr, _amt_key). Among ties prefer same-day.

def rule_narr_exact(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    merged = (
        lo.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]],
            on=["_norm_narr", "_amt_key"],
            suffixes=("_l", "_s"),
        )
    )
    if merged.empty:
        return 0

    # Prefer same-date matches: sort so same-day rows come first
    merged["_same_day"] = (merged["_date_l"] == merged["_date_s"]).astype(int)
    merged = merged.sort_values("_same_day", ascending=False)

    used_l, used_s, matched = set(), set(), 0
    for row in merged.itertuples(index=False):
        il, is_ = int(row.index_l), int(row.index_s)
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_exact, "R1 Narration Exact")


  R1 Narration Exact                                         1 match(es)  █


1

In [8]:
# ── CELL 8 | RULE 2 — Narration Partial  [O(p·w) candidate-only scoring] ─────
# Join on _amt_key → shrinks candidates from m×n to p (amount matches only).
# word_overlap_ratio called only on p pairs, not all m×n.

def rule_narr_partial(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_norm_narr", "_amt_key"]],
            on="_amt_key", suffixes=("_l", "_s"),
        )
        .rename(columns={"_norm_narr_l": "narr_l", "_norm_narr_s": "narr_s"})
    )
    if candidates.empty:
        return 0

    candidates["_score"] = [
        word_overlap_ratio(r.narr_l, r.narr_s)
        for r in candidates.itertuples(index=False)
    ]
    candidates = (candidates[candidates["_score"] >= PARTIAL_MIN_RATIO]
                  .sort_values("_score", ascending=False))

    used_l, used_s, matched = set(), set(), 0
    li_pos = candidates.columns.get_loc("index_l")
    si_pos = candidates.columns.get_loc("index_s")
    for row in candidates.itertuples(index=False):
        il, is_ = int(row[li_pos]), int(row[si_pos])
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_partial, "R2 Narration Partial")


  R2 Narration Partial                                      17 match(es)  █████████████████


17

In [9]:
# ── CELL 9 | RULE 3 — Only Date Exact Match  [OPTIMISED: O(m+n) hash-join] ──
#
# Old approach: nested loop — for every ledger row scan every statement row.
#   Complexity: O(m × n)  →  2977 × 250 = ~744,000 comparisons
#
# New approach: pd.merge on (_date, _amount_rounded) — hash-join.
#   Both frames are hashed once on the join keys, then looked up in O(1).
#   Complexity: O(m + n)  →  ~3,227 operations
#
# Greedy 1-to-1 assignment (highest-confidence first) prevents one statement
# row being claimed by multiple ledger rows.

def rule_date_exact(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()

    # Round amount to 2 dp so it acts as a stable join key
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    # Hash-join on date + rounded amount  →  O(m + n)
    merged = (
        lo.reset_index()[["index", "_date", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_amt_key"]],
            on=["_date", "_amt_key"],
            suffixes=("_l", "_s"),
        )
        .rename(columns={"index_l": "idx_led", "index_s": "idx_stm"})
    )

    if merged.empty:
        return 0

    # Greedy 1-to-1: iterate candidates, mark used on both sides
    used_l, used_s, matched = set(), set(), 0
    for row in merged.itertuples(index=False):
        il, is_ = int(row.idx_led), int(row.idx_stm)
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il)
            used_s.add(is_)
            matched += 1

    return matched

run_rule(rule_date_exact, "R3 Date Exact")


  R3 Date Exact                                            145 match(es)  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████


145

In [10]:
# ── CELL 10 | RULE 4 — Date Inside Range  [O(p log p)] ──────────────────────
# Join on _amt_key → candidate pairs. Vectorised date-diff. Filter ≤ N days.
# Sort by diff (closest first) → greedy 1-to-1.

def rule_date_inside_range(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_date", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_amt_key"]],
            on="_amt_key",
            suffixes=("_l", "_s"),
        )
    )
    if candidates.empty:
        return 0

    # Vectorised date diff — no Python loop
    candidates["_diff"] = (
        (candidates["_date_l"] - candidates["_date_s"]).abs().dt.days
    )
    candidates = candidates[candidates["_diff"] <= DATE_RANGE_DAYS]
    candidates = candidates.sort_values("_diff")   # closest date first

    used_l, used_s, matched = set(), set(), 0
    for row in candidates.itertuples(index=False):
        il, is_ = int(row.index_l), int(row.index_s)
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_date_inside_range, "R4 Date Exact + Inside Range")


  R4 Date Exact + Inside Range                               1 match(es)  █


1

In [11]:
# ── CELL 11 | RULE 5 — Date Any Range  [O(p log p)] ─────────────────────────
# Same as R4 but no date cap — catches anything left with a matching amount.
# Sorted by date diff so closest date still gets priority in greedy assign.

def rule_date_any(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_date", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_amt_key"]],
            on="_amt_key",
            suffixes=("_l", "_s"),
        )
    )
    if candidates.empty:
        return 0

    candidates["_diff"] = (
        (candidates["_date_l"] - candidates["_date_s"]).abs().dt.days
    )
    candidates = candidates.sort_values("_diff")   # closest date first

    used_l, used_s, matched = set(), set(), 0
    for row in candidates.itertuples(index=False):
        il, is_ = int(row.index_l), int(row.index_s)
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_date_any, "R5 Date Any Range")


  R5 Date Any Range                                         11 match(es)  ███████████


11

In [12]:
# ── CELL 12 | RULE 6 — Narration Exact & Date Exact  [O(m+n) hash-join] ─────
# Join on all three keys at once: date + narration + amount.
# Every matched row is an exact hit on all dimensions.

def rule_narr_exact_date_exact(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    merged = (
        lo.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]],
            on=["_date", "_norm_narr", "_amt_key"],
            suffixes=("_l", "_s"),
        )
    )
    if merged.empty:
        return 0

    used_l, used_s, matched = set(), set(), 0
    for row in merged.itertuples(index=False):
        il, is_ = int(row.index_l), int(row.index_s)
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_exact_date_exact, "R6 Narration Exact & Date Exact")


  R6 Narration Exact & Date Exact                            0 match(es)  


0

In [13]:
# ── CELL 13 | RULE 7 — Narration Partial & Date Exact  [O(p·w)] ──────────────
# Join on (_date, _amt_key) → only same-date+amount pairs become candidates.
# word_overlap_ratio scored only on those p pairs.

def rule_narr_partial_date_exact(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]],
            on=["_date", "_amt_key"], suffixes=("_l", "_s"),
        )
        .rename(columns={"_norm_narr_l": "narr_l", "_norm_narr_s": "narr_s"})
    )
    if candidates.empty:
        return 0

    candidates["_score"] = [
        word_overlap_ratio(r.narr_l, r.narr_s)
        for r in candidates.itertuples(index=False)
    ]
    candidates = (candidates[candidates["_score"] >= PARTIAL_MIN_RATIO]
                  .sort_values("_score", ascending=False))

    used_l, used_s, matched = set(), set(), 0
    li_pos = candidates.columns.get_loc("index_l")
    si_pos = candidates.columns.get_loc("index_s")
    for row in candidates.itertuples(index=False):
        il, is_ = int(row[li_pos]), int(row[si_pos])
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_partial_date_exact, "R7 Narration Partial & Date Exact")


  R7 Narration Partial & Date Exact                          0 match(es)  


0

In [14]:
# ── CELL 14 | RULE 8 — Narration Exact & Date Inside Range  [O(p log p)] ─────
# Join on (_norm_narr, _amt_key) → exact narration + amount match.
# Vectorised date diff filter, sorted closest-first.

def rule_narr_exact_date_inside(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]],
            on=["_norm_narr", "_amt_key"],
            suffixes=("_l", "_s"),
        )
    )
    if candidates.empty:
        return 0

    candidates["_diff"] = (
        (candidates["_date_l"] - candidates["_date_s"]).abs().dt.days
    )
    candidates = candidates[candidates["_diff"] <= DATE_RANGE_DAYS]
    candidates = candidates.sort_values("_diff")

    used_l, used_s, matched = set(), set(), 0
    for row in candidates.itertuples(index=False):
        il, is_ = int(row.index_l), int(row.index_s)
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_exact_date_inside, "R8 Narration Exact & Date Inside Range")


  R8 Narration Exact & Date Inside Range                     0 match(es)  


0

In [15]:
# ── CELL 15 | RULE 9 — Narration Partial & Date Inside Range  [O(p·w)] ───────
# Join on _amt_key → filter by date range → score narration on survivors only.
# Date filter runs first (cheap, vectorised) before the word-overlap scoring.

def rule_narr_partial_date_inside(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_date", "_norm_narr", "_amt_key"]],
            on="_amt_key", suffixes=("_l", "_s"),
        )
    )
    if candidates.empty:
        return 0

    # Vectorised date filter first — cheap, shrinks p before word-overlap
    candidates["_diff"] = (candidates["_date_l"] - candidates["_date_s"]).abs().dt.days
    candidates = candidates[candidates["_diff"] <= DATE_RANGE_DAYS]
    if candidates.empty:
        return 0

    candidates = candidates.rename(
        columns={"_norm_narr_l": "narr_l", "_norm_narr_s": "narr_s"}
    )
    candidates["_narr_score"] = [
        word_overlap_ratio(r.narr_l, r.narr_s)
        for r in candidates.itertuples(index=False)
    ]
    candidates = candidates[candidates["_narr_score"] >= PARTIAL_MIN_RATIO]
    candidates["_score"] = (
        candidates["_narr_score"]
        + (DATE_RANGE_DAYS - candidates["_diff"]) / DATE_RANGE_DAYS
    )
    candidates = candidates.sort_values("_score", ascending=False)

    used_l, used_s, matched = set(), set(), 0
    li_pos = candidates.columns.get_loc("index_l")
    si_pos = candidates.columns.get_loc("index_s")
    for row in candidates.itertuples(index=False):
        il, is_ = int(row[li_pos]), int(row[si_pos])
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_partial_date_inside, "R9 Narration Partial & Date Inside Range")


  R9 Narration Partial & Date Inside Range                   0 match(es)  


0

In [16]:
# ── CELL 16 | RULE 10 — Narration Partial & Date Any Range  [O(p·w)] ─────────
# Join on _amt_key → score all amount-matching pairs on narration overlap.
# No date cap — loosest rule, catches everything remaining.

def rule_narr_partial_date_any(rule_name):
    lo = get_open_led().copy()
    so = get_open_stm().copy()
    lo["_amt_key"] = lo["_amount"].round(2)
    so["_amt_key"] = so["_amount"].round(2)

    candidates = (
        lo.reset_index()[["index", "_norm_narr", "_amt_key"]]
        .merge(
            so.reset_index()[["index", "_norm_narr", "_amt_key"]],
            on="_amt_key", suffixes=("_l", "_s"),
        )
        .rename(columns={"_norm_narr_l": "narr_l", "_norm_narr_s": "narr_s"})
    )
    if candidates.empty:
        return 0

    candidates["_score"] = [
        word_overlap_ratio(r.narr_l, r.narr_s)
        for r in candidates.itertuples(index=False)
    ]
    candidates = (candidates[candidates["_score"] >= PARTIAL_MIN_RATIO]
                  .sort_values("_score", ascending=False))

    used_l, used_s, matched = set(), set(), 0
    li_pos = candidates.columns.get_loc("index_l")
    si_pos = candidates.columns.get_loc("index_s")
    for row in candidates.itertuples(index=False):
        il, is_ = int(row[li_pos]), int(row[si_pos])
        if il not in used_l and is_ not in used_s:
            record_match(il, is_, rule_name)
            used_l.add(il); used_s.add(is_)
            matched += 1
    return matched

run_rule(rule_narr_partial_date_any, "R10 Narration Partial & Date Any Range")


  R10 Narration Partial & Date Any Range                     0 match(es)  


0

In [17]:
# ── CELL 17 | ▶ RUN ALL 10 RULES — Pipeline ──────────────────────────────────
# Rules run strictest → loosest.
# A matched row is skipped by all later rules.

import time

reset_all()

PIPELINE = [
    (rule_narr_exact_date_exact,   "R6  Narration Exact & Date Exact         [strictest]"),
    (rule_narr_exact,              "R1  Narration Exact                                  "),
    (rule_date_exact,              "R3  Date Exact                                        "),
    (rule_narr_exact_date_inside,  "R8  Narration Exact & Date Inside Range              "),
    (rule_narr_partial_date_exact, "R7  Narration Partial & Date Exact                   "),
    (rule_narr_partial_date_inside,"R9  Narration Partial & Date Inside Range            "),
    (rule_date_inside_range,       "R4  Date Inside Range                                "),
    (rule_narr_partial_date_any,   "R10 Narration Partial & Date Any Range               "),
    (rule_narr_partial,            "R2  Narration Partial                                "),
    (rule_date_any,                "R5  Date Any Range                    [loosest]      "),
]

print(f"{'Rule':<62} {'Matches':>7}   Time")
print("─" * 80)

rule_log = []
t_total  = time.perf_counter()

for fn, label in PIPELINE:
    t0   = time.perf_counter()
    n    = run_rule(fn, label)
    secs = time.perf_counter() - t0
    rule_log.append({"Rule": label.strip(), "Matches": n, "Time (s)": round(secs, 2)})

total_time  = time.perf_counter() - t_total
led_matched = int(led["_matched"].sum())
stm_matched = int(stm["_matched"].sum())
led_pct     = round(led_matched / len(led) * 100, 1) if len(led) else 0
stm_pct     = round(stm_matched / len(stm) * 100, 1) if len(stm) else 0

print("─" * 80)
print(f"  Total time        : {total_time:.1f}s")
print(f"  Ledger  matched   : {led_matched} / {len(led)}  ({led_pct}%)")
print(f"  Statement matched : {stm_matched} / {len(stm)}  ({stm_pct}%)")

pd.DataFrame(rule_log)


✅ Reset — 2977 ledger / 250 statement rows all open
Rule                                                           Matches   Time
────────────────────────────────────────────────────────────────────────────────
  R6  Narration Exact & Date Exact         [strictest]       0 match(es)  
  R1  Narration Exact                                        1 match(es)  █
  R3  Date Exact                                           161 match(es)  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  R8  Narration Exact & Date Inside Range                    0 match(es)  
  R7  Narration Partial & Date Exact                         0 match(es)  
  R9  Narration Partial & Date Inside Range                  1 match(es)  █
  R4  Date Inside Range                                      1 match(es)  █
  R10 Narration Partial & Date Any Range                     1 match(es)  █
  R2  Narration Partial 

,Rule,Matches,Time (s)
0,R6 Narration Exact & Date Exact [stri...,0,0.02
1,R1 Narration Exact,1,0.02
2,R3 Date Exact,161,0.04
3,R8 Narration Exact & Date Inside Range,0,0.02
4,R7 Narration Partial & Date Exact,0,0.01
5,R9 Narration Partial & Date Inside Range,1,0.03
6,R4 Date Inside Range,1,0.02
7,R10 Narration Partial & Date Any Range,1,0.02
8,R2 Narration Partial,0,0.02
9,R5 Date Any Range [loosest],10,0.03


In [18]:
# ── CELL 18 | Many-to-One & One-to-Many Matching ─────────────────────────────
#
# Runs AFTER all 1-to-1 rules.  Picks up cases where:
#
#   Many-to-One : multiple LEDGER rows sum to one STATEMENT row
#                 e.g. 3 salary splits in ledger → 1 bulk transfer in bank
#
#   One-to-Many : one LEDGER row equals the sum of multiple STATEMENT rows
#                 e.g. 1 combined ledger entry → several bank credits
#
# Algorithm (adapted from reference notebook):
#   1. Iterate every unmatched "one-side" row as the TARGET.
#   2. Collect unmatched "many-side" candidates:
#      - same Dr/Cr side  AND
#      - each individual amount ≤ target amount (can't overshoot alone)
#      - candidate pool capped at 50 to stay fast
#   3. Greedy subset: sort candidates by amount desc, accumulate until
#      running sum ≈ target (within AMOUNT_TOLERANCE).
#   4. Require at least 2 candidates — otherwise it's just a 1-to-1 match.
#   5. Tag all matched rows with shared _group_id so reports can group them.

from itertools import combinations

# ── Ensure group-id column exists (1-to-1 rules leave it None) ───────────────
if "_group_id" not in led.columns:
    led["_group_id"] = None
if "_group_id" not in stm.columns:
    stm["_group_id"] = None

_group_counter = int(led["_matched"].sum() + stm["_matched"].sum()) + 1  # continues from 1-to-1

# ── Core greedy subset function ───────────────────────────────────────────────
def greedy_subset(candidates: pd.DataFrame, target: float, tol: float):
    """
    Sort candidates by amount descending, greedily accumulate.
    Returns list of df indices whose amounts sum within tol of target,
    or None if no valid subset found.
    """
    total, selected = 0.0, []
    for idx in candidates["_amount"].sort_values(ascending=False).index:
        amt = float(candidates.at[idx, "_amount"])
        if abs(total + amt) <= abs(target) + tol:
            total += amt
            selected.append(idx)
            if abs(total - target) <= tol:
                return selected   # found exact-enough subset
    return None if abs(total - target) > tol else selected

# ── Group matching engine ─────────────────────────────────────────────────────
def run_group_matching(direction: str):
    """
    direction = 'Many-to-One'  →  many led rows sum to one stm row
    direction = 'One-to-Many'  →  one led row  = sum of many stm rows
    """
    global _group_counter
    matches = []

    if direction == "Many-to-One":
        target_df, source_df = stm, led          # iterate stm as 'one'
        src_ptr, tgt_ptr = "_stmt_idx", "_led_idx"
    else:
        target_df, source_df = led, stm          # iterate led as 'one'
        src_ptr, tgt_ptr = "_led_idx", "_stmt_idx"

    for ti, trow in target_df[~target_df["_matched"]].iterrows():
        target_amt  = float(trow["_amount"])
        target_side = trow["_side"]

        # Candidates: unmatched, same side, each amount ≤ target
        mask = (
            (~source_df["_matched"]) &
            (source_df["_side"]    == target_side) &
            (source_df["_amount"]  <= target_amt + AMOUNT_TOLERANCE)
        )
        candidates = source_df[mask]

        # Skip if too few (need ≥2 for a group) or too many (cap at 50)
        if len(candidates) < 2 or len(candidates) > 50:
            continue

        group = greedy_subset(candidates, target_amt, AMOUNT_TOLERANCE)

        if group is None or len(group) < 2:
            continue

        gid = _group_counter
        _group_counter += 1

        # ── Tag ALL source rows ───────────────────────────────────────────────
        for src_idx in group:
            source_df.at[src_idx, "_matched"]  = True
            source_df.at[src_idx, "_rule"]     = direction
            source_df.at[src_idx, "_group_id"] = gid
            source_df.at[src_idx, src_ptr]     = ti   # each many-side row → same one-side row

        # ── Tag the target row ────────────────────────────────────────────────
        target_df.at[ti, "_matched"]  = True
        target_df.at[ti, "_rule"]     = direction
        target_df.at[ti, "_group_id"] = gid
        target_df.at[ti, tgt_ptr]     = group[0]   # primary pointer (first src row)

        group_sum = sum(float(source_df.at[i, "_amount"]) for i in group)
        matches.append({
            "GroupId"         : gid,
            "Rule"            : direction,
            "Ledger_Indices"  : group  if direction == "Many-to-One" else [ti],
            "Stmt_Indices"    : [ti]   if direction == "Many-to-One" else group,
            "Target_Amount"   : round(target_amt, 2),
            "Group_Sum"       : round(group_sum,  2),
            "Amt_Diff"        : round(abs(target_amt - group_sum), 2),
            "N_Ledger"        : len(group) if direction == "Many-to-One" else 1,
            "N_Stmt"          : 1          if direction == "Many-to-One" else len(group),
        })

    return matches

# ── Run both directions ───────────────────────────────────────────────────────
print(f"Before group matching → Ledger unmatched: {(~led['_matched']).sum()}  "
      f"Statement unmatched: {(~stm['_matched']).sum()}")
print()

m2o_matches = run_group_matching("Many-to-One")
o2m_matches = run_group_matching("One-to-Many")

all_group_matches = m2o_matches + o2m_matches

print(f"Many-to-One groups found  : {len(m2o_matches)}")
if m2o_matches:
    df_m2o = pd.DataFrame(m2o_matches)
    print(f"  Ledger rows consumed     : {df_m2o['N_Ledger'].sum()}")
    display(df_m2o.head(5))

print()
print(f"One-to-Many groups found  : {len(o2m_matches)}")
if o2m_matches:
    df_o2m = pd.DataFrame(o2m_matches)
    print(f"  Statement rows consumed  : {df_o2m['N_Stmt'].sum()}")
    display(df_o2m.head(5))

print()
print(f"After  group matching → Ledger unmatched: {(~led['_matched']).sum()}  "
      f"Statement unmatched: {(~stm['_matched']).sum()}")


Before group matching → Ledger unmatched: 2802  Statement unmatched: 75

Many-to-One groups found  : 4
  Ledger rows consumed     : 9


,GroupId,Rule,Ledger_Indices,Stmt_Indices,Target_Amount,Group_Sum,Amt_Diff,N_Ledger,N_Stmt
0,351,Many-to-One,"[2864, 134]",[69],33.66,33.66,0.00,2,1
1,352,Many-to-One,"[127, 137, 2863]",[159],27.86,28.00,0.14,3,1
2,353,Many-to-One,"[884, 219]",[189],12.22,12.22,0.00,2,1
3,354,Many-to-One,"[1829, 222]",[231],12.22,12.22,0.00,2,1



One-to-Many groups found  : 5
  Statement rows consumed  : 16


,GroupId,Rule,Ledger_Indices,Stmt_Indices,Target_Amount,Group_Sum,Amt_Diff,N_Ledger,N_Stmt
0,355,One-to-Many,[41],"[104, 172, 234]",2211.86,2211.42,0.44,1,3
1,356,One-to-Many,[147],"[162, 248, 7, 13]",2258.77,2259.27,0.50,1,4
2,357,One-to-Many,[231],"[247, 106, 108, 68]",2258.77,2259.27,0.50,1,4
3,358,One-to-Many,[478],"[83, 174, 188]",5330.10,5330.03,0.07,1,3
4,359,One-to-Many,[1459],"[54, 36]",5226.20,5226.05,0.15,1,2



After  group matching → Ledger unmatched: 2788  Statement unmatched: 55


In [19]:
# ── CELL 19 | Summary Table ───────────────────────────────────────────────────
AUTO_RULES  = ["R6", "R1", "R3"]
GROUP_RULES = ["Many-to-One", "One-to-Many"]

def make_summary(df, doc_type):
    mat   = df[df["_matched"]]
    auto  = mat[mat["_rule"].str.contains("|".join(AUTO_RULES),  na=False)]
    grp   = mat[mat["_rule"].str.contains("|".join(GROUP_RULES), na=False)]
    koff  = mat[~mat["_rule"].str.contains(
                "|".join(AUTO_RULES + GROUP_RULES), na=False)]
    umat  = df[~df["_matched"]]
    return {
        "Document Type"     : doc_type,
        "Total"             : len(df),
        "Automatic (1:1)"   : len(auto),
        "Knockoff (1:1)"    : len(koff),
        "Many-to-One rows"  : len(mat[mat["_rule"] == "Many-to-One"]),
        "One-to-Many rows"  : len(mat[mat["_rule"] == "One-to-Many"]),
        "Manual"            : 0,
        "Unmatched"         : len(umat),
        "Matched %"         : round(len(mat)/len(df)*100, 1) if len(df) else 0,
    }

summary = pd.DataFrame([
    make_summary(led, "Ledger"),
    make_summary(stm, "Statement"),
])
display(summary)


,Document Type,Total,Automatic (1:1),Knockoff (1:1),Many-to-One rows,One-to-Many rows,Manual,Unmatched,Matched %
0,Ledger,2977,163,12,9,5,0,2788,6.3
1,Statement,250,163,12,4,16,0,55,78.0


In [20]:
# ── CELL 19 | Matched Pairs — Which Row Matched Which ────────────────────────
rows = []
for li, lrow in led[led["_matched"]].iterrows():
    si   = lrow["_stmt_idx"]
    if pd.isna(si): continue
    si   = int(si)
    srow = stm.loc[si]
    ld, sd = lrow["_date"], srow["_date"]
    delta  = abs((ld - sd).days) if pd.notna(ld) and pd.notna(sd) else ""
    rows.append({
        "Rule"         : lrow["_rule"],
        "Ledger Idx"   : li,
        "Ledger Date"  : ld.date() if pd.notna(ld) else "",
        "Ledger Narr"  : lrow["_narration"][:50],
        "Ledger Amt"   : lrow["_amount"],
        "Ledger Side"  : lrow["_side"],
        "Stmt Idx"     : si,
        "Stmt Date"    : sd.date() if pd.notna(sd) else "",
        "Stmt Narr"    : srow["_narration"][:50],
        "Stmt Amt"     : srow["_amount"],
        "Stmt Side"    : srow["_side"],
        "Date Δ days"  : delta,
        "Narr Overlap" : round(word_overlap_ratio(lrow["_norm_narr"], srow["_norm_narr"]), 2),
    })

df_pairs = pd.DataFrame(rows)
print(f"Total matched pairs: {len(df_pairs)}")
display(df_pairs)


Total matched pairs: 189


,Rule,Ledger Idx,Ledger Date,Ledger Narr,Ledger Amt,Ledger Side,Stmt Idx,Stmt Date,Stmt Narr,Stmt Amt,Stmt Side,Date Δ days,Narr Overlap
0,One-to-Many,41,2021-03-26,,2211.86,Dr,104,2021-01-05,469 / MISCELLANEOUS ACH DEBIT EXPERTPAY EXPERT...,1474.09,Dr,80,0.0
1,R3 Date Exact ...,47,2021-03-30,EFT received against invoice p,756.69,Dr,169,2021-03-30,169 / MISCELLANEOUS ACH CREDIT GA0676 COFORGE ...,756.69,Cr,0,0.0
2,R3 Date Exact ...,50,2021-03-30,Payment made against Inv due,129.34,Dr,235,2021-03-30,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,129.34,Dr,0,0.0
3,R3 Date Exact ...,51,2021-03-30,Payment made against Inv due,66.00,Dr,109,2021-03-30,469 / MISCELLANEOUS ACH DEBIT NAVIA BENEFIT SO...,66.00,Dr,0,0.0
4,R3 Date Exact ...,52,2021-03-30,Payment made against Inv due,30.74,Dr,22,2021-03-30,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,30.74,Dr,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,R3 Date Exact ...,2870,2021-02-17,,149795.95,Dr,53,2021-02-17,495 / OUTGOING MONEY TRANSFER WT 210216-171498...,149795.95,Dr,0,0.0
185,R3 Date Exact ...,2945,2021-03-12,Payment made against payment,531.23,Dr,151,2021-03-12,469 / MISCELLANEOUS ACH DEBIT PECO ENERGY COMP...,531.23,Dr,0,0.0
186,R3 Date Exact ...,2946,2021-03-12,Payment made against payment,438.61,Dr,152,2021-03-12,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,438.61,Dr,0,0.0
187,R3 Date Exact ...,2947,2021-03-12,Payment made against payment,385.55,Dr,155,2021-03-12,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,385.55,Dr,0,0.0


In [21]:
# ── CELL 20 | Unmatched Rows ─────────────────────────────────────────────────
df_unled = led[~led["_matched"]][["_date", "_narration", "_amount", "_side",
                                    L_DOC_TYPE, L_DOC_NO, L_CHEQUE]].copy()
df_unled.columns = ["Date","Narration","Amount","Side","Doc Type","Doc No","Cheque No"]

df_unstm = stm[~stm["_matched"]][["_date", "_narration", "_amount", "_side",
                                     S_UNIQUE_ID, S_BANK_REF, S_CUST_REF]].copy()
df_unstm.columns = ["Date","Narration","Amount","Side","Unique ID","Bank Ref","Cust Ref"]

print(f"Unmatched Ledger    : {len(df_unled)}")
print(f"Unmatched Statement : {len(df_unstm)}")
print()
print("── Unmatched Ledger (first 20) ─────────────────────────────────────────")
display(df_unled.head(20))
print()
print("── Unmatched Statement (first 20) ──────────────────────────────────────")
display(df_unstm.head(20))


Unmatched Ledger    : 2788
Unmatched Statement : 55

── Unmatched Ledger (first 20) ─────────────────────────────────────────


,Date,Narration,Amount,Side,Doc Type,Doc No,Cheque No
0,2021-03-26,,2620.28,Dr,KZ,601244,NaN
1,2021-03-26,,786.79,Dr,KZ,601244,NaN
2,2021-03-26,,810.56,Dr,KZ,601244,NaN
3,2021-03-26,,47.77,Dr,KZ,601244,NaN
4,2021-03-26,,1039.42,Dr,KZ,601244,NaN
5,2021-03-26,,596.13,Dr,KZ,601244,NaN
6,2021-03-26,,668.88,Dr,KZ,601244,NaN
7,2021-03-26,,643.85,Dr,KZ,601244,NaN
8,2021-03-16,PAID AGAINST EE EXPENSE REIMB,75.00,Dr,KZ,601195,NaN
9,2021-03-16,PAID AGAINST EE EXPENSE REIMB,75.00,Dr,KZ,601195,NaN



── Unmatched Statement (first 20) ──────────────────────────────────────


,Date,Narration,Amount,Side,Unique ID,Bank Ref,Cust Ref
1,2021-01-19,469 / MISCELLANEOUS ACH DEBIT ACH ORIGINATION ...,539182.25,Dr,NaN,IA000012026893,00000000000
2,2021-01-19,469 / MISCELLANEOUS ACH DEBIT ACH ORIGINATION ...,12582.98,Dr,NaN,IA000016600398,00000000000
8,2021-03-11,469 / MISCELLANEOUS ACH DEBIT ACH ORIGINATION ...,1977629.02,Dr,NaN,IA000015040468,00000000000
9,2021-03-10,208 / INTL MONEY TRANSFER CREDIT WT FED#04865 ...,2999975.00,Cr,RG210310043165,IA009968720755,00000000000
12,2021-03-09,469 / MISCELLANEOUS ACH DEBIT AMERICAN EXPRESS...,33845.61,Dr,00000091004358076949,IA000010229372,00000000000
15,2021-01-14,208 / INTL MONEY TRANSFER CREDIT WT FED#04081 ...,1999975.00,Cr,RG210114026583,IA009948778897,00000000000
16,2021-01-14,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,157629.57,Dr,91004053679433,IA000097133731,00000000000
17,2021-01-14,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,68320.13,Dr,91004053679434,IA000097133734,00000000000
18,2021-01-14,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,51814.63,Dr,91004053679430,IA000097105169,00000000000
19,2021-01-14,469 / MISCELLANEOUS ACH DEBIT GA0676 COFORGE L...,18834.87,Dr,91004053679431,IA000097105172,00000000000


In [22]:
# ── CELL 21 | Export to Excel ────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

H_FONT = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
H_FILL = PatternFill("solid", start_color="1F4E79")
H_ALGN = Alignment(horizontal="center", vertical="center", wrap_text=True)
B_FONT = Font(name="Calibri", size=9)
B_ALGN = Alignment(horizontal="center", vertical="center")
GRN    = PatternFill("solid", start_color="E2EFDA")
RED    = PatternFill("solid", start_color="FCE4D6")
ALT    = PatternFill("solid", start_color="DDEBF7")

def safe_val(v):
    if v is None: return ""
    if isinstance(v, float) and np.isnan(v): return ""
    if str(v) in ("<NA>", "NaT", "nan", "None"): return ""
    return v

def write_sheet(ws, df, fill_fn=None):
    ws.freeze_panes = "A2"
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(1, ci, col)
        c.font = H_FONT; c.fill = H_FILL; c.alignment = H_ALGN
    for ri, tup in enumerate(df.itertuples(index=False), 2):
        fill = fill_fn(tup, ri) if fill_fn else (ALT if ri % 2 == 0 else None)
        for ci, val in enumerate(tup, 1):
            c = ws.cell(ri, ci, safe_val(val))
            c.font = B_FONT; c.alignment = B_ALGN
            if fill: c.fill = fill
    for col in ws.columns:
        mx = max((len(str(c.value or "")) for c in col), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(mx + 3, 55)

wb = Workbook()

# 1 — Summary
ws1 = wb.active; ws1.title = "Summary"
write_sheet(ws1, summary)

# 2 — Rule Log
ws2 = wb.create_sheet("Rule Log")
write_sheet(ws2, pd.DataFrame(rule_log))

# 3 — Matched Pairs (green)
ws3 = wb.create_sheet("Matched Pairs")
write_sheet(ws3, df_pairs, fill_fn=lambda r, i: GRN)

# 4 — Unmatched Ledger (red)
ws4 = wb.create_sheet("Unmatched Ledger")
write_sheet(ws4, df_unled, fill_fn=lambda r, i: RED)

# 5 — Unmatched Statement (red)
ws5 = wb.create_sheet("Unmatched Statement")
write_sheet(ws5, df_unstm, fill_fn=lambda r, i: RED)

# 6 — Full Ledger with status
full_led = led[[
    "_date", L_DOC_TYPE, L_DOC_NO, L_CHEQUE,
    "_narration", "_amount", "_side", L_BANK_GL,
    "_matched", "_rule", "_stmt_idx",
]].rename(columns={
    "_date":"Date", "_narration":"Narration", "_amount":"Amount",
    "_side":"Dr/Cr", L_DOC_TYPE:"Doc Type", L_DOC_NO:"Doc No",
    L_CHEQUE:"Cheque No", L_BANK_GL:"Bank GL",
    "_matched":"Matched?", "_rule":"Rule", "_stmt_idx":"Stmt Row",
})
ws6 = wb.create_sheet("Full Ledger")
write_sheet(ws6, full_led,
            fill_fn=lambda r, i: GRN if r[8] else RED)

# 7 — Full Statement with status
full_stm = stm[[
    "_date", S_UNIQUE_ID, S_BANK_REF, S_CUST_REF,
    "_narration", "_amount", "_side",
    "_matched", "_rule", "_led_idx",
]].rename(columns={
    "_date":"Date", "_narration":"Narration", "_amount":"Amount",
    "_side":"Dr/Cr", S_UNIQUE_ID:"Unique ID", S_BANK_REF:"Bank Ref",
    S_CUST_REF:"Cust Ref", "_matched":"Matched?",
    "_rule":"Rule", "_led_idx":"Ledger Row",
})
ws7 = wb.create_sheet("Full Statement")
write_sheet(ws7, full_stm,
            fill_fn=lambda r, i: GRN if r[7] else RED)

OUT = "Reconciliation_Output.xlsx"
wb.save(OUT)
print(f"✅  Saved → {OUT}  ({len(wb.sheetnames)} sheets)")
# In Colab: from google.colab import files; files.download(OUT)


KeyError: "['Bank GL A/C'] not in index"

In [ ]:
# ── CELL 23 | Build Report DataFrames (1:1 + Many-to-One + One-to-Many) ──────
#
# Reconciled   → every matched pair, one row per (ledger_idx, stmt_idx) combo
#                GroupId ties together rows belonging to the same group match
# Knockoff     → individual rows from Knockoff + Group matches (one row each)
# Unreconciled → all still-unmatched rows from both files

recon_rows    = []
knockoff_rows = []
uid = 1

# ─────────────────────────────────────────────────────────────────────────────
# Helper: build one reconciled output row given a led and stm row
# ─────────────────────────────────────────────────────────────────────────────
def make_recon_row(li, lrow, si, srow, uid, group_id, match_type, comments, status):
    ld = lrow["_date"]
    sd = srow["_date"]
    return {
        "UniqueId"               : uid,
        "GroupId"                : group_id,
        "FK_InputId"             : si,
        "FK_InputId_df2"         : li,
        "Entity"                 : "",
        "EntityName"             : "",
        "GL_Code"                : "",
        "GL_Description"         : "",
        "TransactionDate"        : ld.strftime("%d-%m-%Y") if pd.notna(ld) else "",
        "FlexDate1"              : "",
        "FlexDate2"              : "",
        "KeyIdentifier1"         : srow.get(S_UNIQUE_ID, ""),
        "KeyIdentifier2"         : srow.get(S_BANK_REF,  ""),
        "KeyIdentifier3"         : srow.get(S_CUST_REF,  ""),
        "Narration1"             : srow.get(S_NARRATION,  ""),
        "Narration2"             : srow.get(S_NARRATION2, ""),
        "Narration3"             : "",
        "TransactionCurrency"    : "",
        "Status"                 : status,
        "Amount"                 : lrow["_amount"],
        "GL_Code_df2"            : lrow.get(L_BANK_GL, ""),
        "GL_Description_df2"     : lrow.get("GL A/c Description", ""),
        "TransactionDate_df2"    : sd.strftime("%d-%m-%Y") if pd.notna(sd) else "",
        "FlexDate1_df2"          : "",
        "FlexDate2_df2"          : "",
        "KeyIdentifier1_df2"     : lrow.get(L_DOC_NO,   ""),
        "KeyIdentifier2_df2"     : lrow.get(L_CHEQUE,   ""),
        "KeyIdentifier3_df2"     : lrow.get(L_DOC_TYPE, ""),
        "Narration1_df2"         : lrow.get(L_DESC,        ""),
        "Narration2_df2"         : lrow.get(L_PARTICULARS, ""),
        "Narration3_df2"         : "",
        "TransactionCurrency_df2": "",
        "Amount_df2"             : srow["_amount"],
        "Amount_Difference"      : round(lrow["_amount"] - srow["_amount"], 2),
        "MatchType"              : match_type,
        "Comments"               : comments,
    }

def make_knockoff_row(row, doc_type):
    d = row["_date"]
    return {
        "Entity"          : "",
        "EntityName"      : "",
        "GL Code"         : row.get(L_BANK_GL, "") if doc_type == "Ledger" else "",
        "GL Description"  : row.get("GL A/c Description", "") if doc_type == "Ledger" else "",
        "TransactionDate" : d.strftime("%d-%m-%Y") if pd.notna(d) else "",
        "FlexDate1"       : "",
        "FlexDate2"       : "",
        "KeyIdentifier1"  : row.get(L_DOC_NO,   "") if doc_type == "Ledger" else row.get(S_UNIQUE_ID, ""),
        "KeyIdentifier2"  : row.get(L_CHEQUE,   "") if doc_type == "Ledger" else row.get(S_BANK_REF,  ""),
        "KeyIdentifier3"  : row.get(L_DOC_TYPE, "") if doc_type == "Ledger" else row.get(S_CUST_REF,  ""),
        "Narration1"      : row.get(L_DESC,        "") if doc_type == "Ledger" else row.get(S_NARRATION,  ""),
        "Narration2"      : row.get(L_PARTICULARS, "") if doc_type == "Ledger" else row.get(S_NARRATION2, ""),
        "Narration3"      : "",
        "Amount"          : row["_amount"],
        "DocumentType"    : doc_type,
        "Bank"            : row.get(L_BANK_GL, "") if doc_type == "Ledger" else row.get("BANK",""),
        "GroupId"         : row.get("_group_id", ""),
        "Remarks"         : f"Matched via {row['_rule']}",
    }

# ─────────────────────────────────────────────────────────────────────────────
# 1.  1-to-1 matched rows
# ─────────────────────────────────────────────────────────────────────────────
ONE_TO_ONE_RULES = {"Many-to-One", "One-to-Many"}

for li, lrow in led[led["_matched"]].iterrows():
    if lrow["_rule"] in ONE_TO_ONE_RULES:
        continue                                  # handled separately below
    si = lrow["_stmt_idx"]
    if pd.isna(si):
        continue
    si   = int(si)
    srow = stm.loc[si]
    rule     = lrow["_rule"]
    status   = get_status(rule)
    recon_rows.append(make_recon_row(
        li, lrow, si, srow,
        uid       = uid,
        group_id  = uid,                          # 1-to-1: GroupId = UniqueId
        match_type= get_match_type(rule),
        comments  = get_comments(rule),
        status    = status,
    ))
    if status == "Knockoff":
        knockoff_rows.append(make_knockoff_row(lrow, "Ledger"))
        knockoff_rows.append(make_knockoff_row(srow, "Statement"))
    uid += 1

# ─────────────────────────────────────────────────────────────────────────────
# 2.  Many-to-One matched rows (many ledger rows → one statement row)
# ─────────────────────────────────────────────────────────────────────────────
for m in (all_group_matches if 'all_group_matches' in dir() else []):
    if m["Rule"] != "Many-to-One":
        continue
    gid  = m["GroupId"]
    si   = m["Stmt_Indices"][0]
    srow = stm.loc[si]
    group_sum = m["Group_Sum"]

    for li in m["Ledger_Indices"]:
        lrow = led.loc[li]
        recon_rows.append(make_recon_row(
            li, lrow, si, srow,
            uid       = uid,
            group_id  = gid,
            match_type= "Many-to-One",
            comments  = (f"Many-to-One: {len(m['Ledger_Indices'])} ledger rows "
                         f"(sum={group_sum}) → 1 statement row (amt={m['Target_Amount']})"),
            status    = "Knockoff",
        ))
        knockoff_rows.append({**make_knockoff_row(lrow, "Ledger"),
                               "GroupId": gid,
                               "Remarks": f"Many-to-One group {gid} — "
                                          f"{len(m['Ledger_Indices'])} ledger rows → 1 stmt row"})
        uid += 1

    # Statement side knockoff row
    knockoff_rows.append({**make_knockoff_row(srow, "Statement"),
                           "GroupId": gid,
                           "Remarks": f"Many-to-One group {gid} — 1 stmt row ← "
                                      f"{len(m['Ledger_Indices'])} ledger rows"})

# ─────────────────────────────────────────────────────────────────────────────
# 3.  One-to-Many matched rows (one ledger row → many statement rows)
# ─────────────────────────────────────────────────────────────────────────────
for m in (all_group_matches if 'all_group_matches' in dir() else []):
    if m["Rule"] != "One-to-Many":
        continue
    gid  = m["GroupId"]
    li   = m["Ledger_Indices"][0]
    lrow = led.loc[li]
    group_sum = m["Group_Sum"]

    for si in m["Stmt_Indices"]:
        srow = stm.loc[si]
        recon_rows.append(make_recon_row(
            li, lrow, si, srow,
            uid       = uid,
            group_id  = gid,
            match_type= "One-to-Many",
            comments  = (f"One-to-Many: 1 ledger row (amt={m['Target_Amount']}) → "
                         f"{len(m['Stmt_Indices'])} statement rows (sum={group_sum})"),
            status    = "Knockoff",
        ))
        knockoff_rows.append({**make_knockoff_row(srow, "Statement"),
                               "GroupId": gid,
                               "Remarks": f"One-to-Many group {gid} — "
                                          f"1 ledger row → {len(m['Stmt_Indices'])} stmt rows"})
        uid += 1

    # Ledger side knockoff row
    knockoff_rows.append({**make_knockoff_row(lrow, "Ledger"),
                           "GroupId": gid,
                           "Remarks": f"One-to-Many group {gid} — 1 ledger row → "
                                      f"{len(m['Stmt_Indices'])} stmt rows"})

# ─────────────────────────────────────────────────────────────────────────────
# 4.  Unreconciled rows (still unmatched after everything)
# ─────────────────────────────────────────────────────────────────────────────
unrecon_rows = []
for li, lrow in led[~led["_matched"]].iterrows():
    unrecon_rows.append({
        "Entity":"","EntityName":"",
        "GL Code"         : lrow.get(L_BANK_GL,""),
        "GL Description"  : lrow.get("GL A/c Description",""),
        "TransactionDate" : lrow["_date"].strftime("%d-%m-%Y") if pd.notna(lrow["_date"]) else "",
        "FlexDate1":"","FlexDate2":"",
        "KeyIdentifier1"  : lrow.get(L_DOC_NO,""),
        "KeyIdentifier2"  : lrow.get(L_CHEQUE,""),
        "KeyIdentifier3"  : lrow.get(L_DOC_TYPE,""),
        "Narration1"      : lrow.get(L_DESC,""),
        "Narration2"      : lrow.get(L_PARTICULARS,""),
        "Narration3"      : "",
        "Amount"          : lrow["_amount"],
        "DocumentType"    : "Ledger",
        "Bank"            : lrow.get(L_BANK_GL,""),
    })

for si, srow in stm[~stm["_matched"]].iterrows():
    unrecon_rows.append({
        "Entity":"","EntityName":"",
        "GL Code":"","GL Description":"",
        "TransactionDate" : srow["_date"].strftime("%d-%m-%Y") if pd.notna(srow["_date"]) else "",
        "FlexDate1":"","FlexDate2":"",
        "KeyIdentifier1"  : srow.get(S_UNIQUE_ID,""),
        "KeyIdentifier2"  : srow.get(S_BANK_REF,""),
        "KeyIdentifier3"  : srow.get(S_CUST_REF,""),
        "Narration1"      : srow.get(S_NARRATION,""),
        "Narration2"      : srow.get(S_NARRATION2,""),
        "Narration3"      : "",
        "Amount"          : srow["_amount"],
        "DocumentType"    : "Statement",
        "Bank"            : srow.get("BANK",""),
    })

# ─────────────────────────────────────────────────────────────────────────────
# 5.  Assemble DataFrames with exact column order
# ─────────────────────────────────────────────────────────────────────────────
RECON_COLS = [
    "UniqueId","GroupId","FK_InputId","FK_InputId_df2",
    "Entity","EntityName",
    "GL_Code","GL_Description","TransactionDate","FlexDate1","FlexDate2",
    "KeyIdentifier1","KeyIdentifier2","KeyIdentifier3",
    "Narration1","Narration2","Narration3","TransactionCurrency","Status","Amount",
    "GL_Code_df2","GL_Description_df2","TransactionDate_df2",
    "FlexDate1_df2","FlexDate2_df2",
    "KeyIdentifier1_df2","KeyIdentifier2_df2","KeyIdentifier3_df2",
    "Narration1_df2","Narration2_df2","Narration3_df2","TransactionCurrency_df2",
    "Amount_df2","Amount_Difference","MatchType","Comments",
]
KNOCKOFF_COLS = [
    "Entity","EntityName","GL Code","GL Description",
    "TransactionDate","FlexDate1","FlexDate2",
    "KeyIdentifier1","KeyIdentifier2","KeyIdentifier3",
    "Narration1","Narration2","Narration3",
    "Amount","DocumentType","Bank","GroupId","Remarks",
]
UNRECON_COLS = [
    "Entity","EntityName","GL Code","GL Description",
    "TransactionDate","FlexDate1","FlexDate2",
    "KeyIdentifier1","KeyIdentifier2","KeyIdentifier3",
    "Narration1","Narration2","Narration3",
    "Amount","DocumentType","Bank",
]

df_reconciled   = pd.DataFrame(recon_rows)
df_knockoff     = pd.DataFrame(knockoff_rows)
df_unreconciled = pd.DataFrame(unrecon_rows)

if not df_reconciled.empty:   df_reconciled   = df_reconciled[RECON_COLS]
if not df_knockoff.empty:     df_knockoff     = df_knockoff[KNOCKOFF_COLS]
if not df_unreconciled.empty: df_unreconciled = df_unreconciled[UNRECON_COLS]

# Summary
m2o_count = len([m for m in (all_group_matches if 'all_group_matches' in dir() else []) if m["Rule"]=="Many-to-One"])
o2m_count = len([m for m in (all_group_matches if 'all_group_matches' in dir() else []) if m["Rule"]=="One-to-Many"])

print(f"Reconciled rows   : {len(df_reconciled)}  (includes group rows)")
print(f"  of which 1:1    : {len(df_reconciled[df_reconciled['MatchType'].str.contains('Exact|Range|Partial', na=False)]) if not df_reconciled.empty else 0}")
print(f"  Many-to-One grps: {m2o_count}")
print(f"  One-to-Many grps: {o2m_count}")
print(f"Knockoff rows     : {len(df_knockoff)}")
print(f"Unreconciled rows : {len(df_unreconciled)}")
print()
display(df_reconciled.head(3))


In [ ]:
# ── CELL 24 | Export — Reconciled / Knockoff / Unreconciled / Group Reports ────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Palette ───────────────────────────────────────────────────────────────────
BLUE_DARK   = "1F4E79"   # header background
BLUE_MID    = "2E75B6"   # sub-header / divider
GREEN_FILL  = "E2EFDA"   # reconciled / automatic rows
AMBER_FILL  = "FFF2CC"   # knockoff rows
RED_FILL    = "FCE4D6"   # unreconciled rows
ALT_FILL    = "DDEBF7"   # alternating row tint

H_FONT  = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
SH_FONT = Font(bold=True, color="FFFFFF", name="Calibri", size=9)
B_FONT  = Font(name="Calibri", size=9)
CTR     = Alignment(horizontal="center", vertical="center", wrap_text=False)
LEFT    = Alignment(horizontal="left",   vertical="center", wrap_text=False)
THIN    = Side(style="thin", color="BDD7EE")

def border():
    return Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def safe(v):
    if v is None: return ""
    if isinstance(v, float) and (v != v): return ""   # NaN
    if str(v) in ("<NA>","NaT","nan","None"): return ""
    return v

def write_report_sheet(ws, df, header_fill_hex, row_fill_fn=None, freeze="B2"):
    """Write df to ws with styled headers and auto-width columns."""
    ws.freeze_panes = freeze

    # ── Header row ────────────────────────────────────────────────────────────
    hfill = PatternFill("solid", start_color=header_fill_hex)
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(1, ci, col)
        c.font = H_FONT
        c.fill = hfill
        c.alignment = CTR
        c.border = border()

    # ── Data rows ─────────────────────────────────────────────────────────────
    for ri, tup in enumerate(df.itertuples(index=False), 2):
        fill = row_fill_fn(tup, ri) if row_fill_fn else (
            PatternFill("solid", start_color=ALT_FILL) if ri % 2 == 0 else None
        )
        for ci, val in enumerate(tup, 1):
            c = ws.cell(ri, ci, safe(val))
            c.font = B_FONT
            c.alignment = LEFT
            c.border = border()
            if fill:
                c.fill = fill

    # ── Auto column width ─────────────────────────────────────────────────────
    for col in ws.columns:
        header_len = len(str(col[0].value or ""))
        data_max   = max((len(str(c.value or "")) for c in col[1:11]), default=0)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(
            max(header_len, data_max) + 3, 40
        )

# ── Status-based row colour for Reconciled sheet ──────────────────────────────
STATUS_COL_IDX = RECON_COLS.index("Status")
def recon_fill(tup, ri):
    status = tup[STATUS_COL_IDX]
    if status == "Automatic":
        return PatternFill("solid", start_color=GREEN_FILL)
    return PatternFill("solid", start_color=AMBER_FILL)

# ── Document-type colour for Knockoff / Unreconciled ─────────────────────────
DOCTYPE_K_IDX = KNOCKOFF_COLS.index("DocumentType")
DOCTYPE_U_IDX = UNRECON_COLS.index("DocumentType")

def knockoff_fill(tup, ri):
    return PatternFill("solid", start_color=AMBER_FILL)

def unrecon_fill(tup, ri):
    dt = tup[DOCTYPE_U_IDX]
    return PatternFill("solid", start_color=RED_FILL)

# ── Build workbook ─────────────────────────────────────────────────────────────
wb_report = Workbook()

# 1 — Reconciled Data
ws_recon = wb_report.active
ws_recon.title = "Reconciled Data"
if not df_reconciled.empty:
    write_report_sheet(ws_recon, df_reconciled,
                       header_fill_hex=BLUE_DARK,
                       row_fill_fn=recon_fill)
else:
    ws_recon.cell(1, 1, "No reconciled rows")

# 2 — Knockoff Data
ws_ko = wb_report.create_sheet("Knockoff Data")
if not df_knockoff.empty:
    write_report_sheet(ws_ko, df_knockoff,
                       header_fill_hex=BLUE_MID,
                       row_fill_fn=knockoff_fill)
else:
    ws_ko.cell(1, 1, "No knockoff rows")

# 3 — Unreconciled Data
ws_un = wb_report.create_sheet("Unreconciled Data")
if not df_unreconciled.empty:
    write_report_sheet(ws_un, df_unreconciled,
                       header_fill_hex=BLUE_DARK,
                       row_fill_fn=unrecon_fill)
else:
    ws_un.cell(1, 1, "No unreconciled rows")

# 4 — Summary tab ──────────────────────────────────────────────────────────────
ws_sum = wb_report.create_sheet("Summary", 0)   # insert first
ws_sum.freeze_panes = "A2"

auto_rows = df_reconciled[df_reconciled["Status"]=="Automatic"] if not df_reconciled.empty else pd.DataFrame()
ko_rows   = df_reconciled[df_reconciled["Status"]=="Knockoff"]  if not df_reconciled.empty else pd.DataFrame()

summary_data = [
    ["Category",          "Ledger",                                "Statement",                           "Total"],
    ["Total Rows",        len(led),                                len(stm),                              len(led)+len(stm)],
    ["Reconciled (Auto)", len(auto_rows),                          len(auto_rows),                        len(auto_rows)],
    ["Reconciled (Knock)",len(ko_rows),                            len(ko_rows),                          len(ko_rows)],
    ["Total Matched",     int(led["_matched"].sum()),              int(stm["_matched"].sum()),             ""],
    ["Unreconciled",      int((~led["_matched"]).sum()),           int((~stm["_matched"]).sum()),          len(df_unreconciled)],
    ["Match %",
     f"{led['_matched'].mean()*100:.1f}%",
     f"{stm['_matched'].mean()*100:.1f}%", ""],
]
hfill_sum = PatternFill("solid", start_color=BLUE_DARK)
for ri, row in enumerate(summary_data, 1):
    for ci, val in enumerate(row, 1):
        c = ws_sum.cell(ri, ci, val)
        c.border = border()
        if ri == 1:
            c.font = H_FONT; c.fill = hfill_sum; c.alignment = CTR
        elif ci == 1:
            c.font = Font(bold=True, name="Calibri", size=9); c.alignment = LEFT
        else:
            c.font = B_FONT; c.alignment = CTR
for col in ws_sum.columns:
    mx = max(len(str(c.value or "")) for c in col)
    ws_sum.column_dimensions[get_column_letter(col[0].column)].width = mx + 4

# 4b — Group Matches detail sheet (Many-to-One and One-to-Many)
ws_grp = wb_report.create_sheet("Group Matches")
if 'all_group_matches' in dir() and all_group_matches:
    grp_rows = []
    for m in all_group_matches:
        for li in m["Ledger_Indices"]:
            lrow = led.loc[li]
            for si in m["Stmt_Indices"]:
                srow = stm.loc[si]
                grp_rows.append({
                    "GroupId"        : m["GroupId"],
                    "Rule"           : m["Rule"],
                    "N_Ledger"       : m["N_Ledger"],
                    "N_Statement"    : m["N_Stmt"],
                    "Target_Amount"  : m["Target_Amount"],
                    "Group_Sum"      : m["Group_Sum"],
                    "Amt_Diff"       : m["Amt_Diff"],
                    "Ledger_Idx"     : li,
                    "Ledger_Date"    : lrow["_date"].strftime("%d-%m-%Y") if pd.notna(lrow["_date"]) else "",
                    "Ledger_Narr"    : lrow.get(L_DESC,"")[:50],
                    "Ledger_Amount"  : lrow["_amount"],
                    "Ledger_Side"    : lrow["_side"],
                    "Stmt_Idx"       : si,
                    "Stmt_Date"      : srow["_date"].strftime("%d-%m-%Y") if pd.notna(srow["_date"]) else "",
                    "Stmt_Narr"      : srow.get(S_NARRATION,"")[:50],
                    "Stmt_Amount"    : srow["_amount"],
                    "Stmt_Side"      : srow["_side"],
                })
    df_grp = pd.DataFrame(grp_rows)
    # Colour Many-to-One amber, One-to-Many blue
    M2O_FILL = PatternFill("solid", start_color="FFF2CC")
    O2M_FILL = PatternFill("solid", start_color="DDEBF7")
    RULE_COL  = df_grp.columns.get_loc("Rule")
    write_report_sheet(ws_grp, df_grp,
                       header_fill_hex=BLUE_MID,
                       row_fill_fn=lambda r, i: M2O_FILL if r[RULE_COL]=="Many-to-One" else O2M_FILL)
    print(f"   Group Matches rows : {len(df_grp)}")
else:
    ws_grp.cell(1, 1, "No group matches found")

OUT_REPORT = "Reconciliation_Report.xlsx"
wb_report.save(OUT_REPORT)
print(f"✅  Report saved → {OUT_REPORT}")
print(f"   Sheets : {wb_report.sheetnames}")
print()
print(f"   Reconciled (Automatic) : {len(auto_rows)} pairs")
print(f"   Reconciled (Knockoff)  : {len(ko_rows)} pairs")
print(f"   Knockoff detail rows   : {len(df_knockoff)}")
print(f"   Unreconciled rows      : {len(df_unreconciled)}")
# In Colab: from google.colab import files; files.download(OUT_REPORT)
